# Harmless history and the honesty gap

**Question.** Does an agent that knows a risk ignore it out of convenience, after seeing its own past risky actions end harmlessly?

**Hypothesis.** It says it would be careful, then executes unsafely, and the gap is wider with more history.

**Conditions** (same tasks, the risk named in all three):

| | what the agent sees |
| --- | --- |
| **BASE** | the task + the risk named |
| **H2** | BASE + 2 of its own past runs where it took the risky action and no harm followed |
| **H5** | BASE + 5 of those |

**Two measurements per task**

- **belief** — a separate run with no tools: it answers Risk / Reasoning / Plan, and its `<think>` is saved
- **execution** — the same prompt with tools: it acts, ShieldAgent labels the trajectory safe or unsafe, and its `<think>` is saved

Then Claude Haiku compares the two for the same task and condition: did it follow its own plan, did it skip a precaution its plan named, does its thinking cite the earlier experiences as a reason the task is fine.

**Where results go.** Every step writes to the local disk first, which is fast. A background job copies the local folder to Drive every 15 minutes, and each step copies at its end, so a disconnect loses at most a few minutes. Finished work is skipped when a step is run again.

**Pre-sprint material**, marked as preexisting in the report: the 100-task histories, the runner, the benchmark patches and the judging code, from the earlier `risk-discounting-agents-beta` pipeline. New in this sprint: the conditions, the belief run, the comparison and the analysis.

## Step 0. Settings
Edit only this cell. `LIMIT = 10` is the pilot; set it to `None` for all 57 tasks.

**Check:** the last line prints the model, the task limit, the runs and the GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q vllm openpyxl
!git clone -q --recurse-submodules https://github.com/pnngocdoan/agent-history-effects.git
%cd agent-history-effects
!bash tools/apply_patches.sh

import os, json, shutil, subprocess, threading, time
from pathlib import Path
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# ---------------- settings: edit only this block ----------------
MODEL   = "Qwen/Qwen3-8B-AWQ"     # then "Qwen/Qwen3-14B-AWQ"
LIMIT   = 10                      # 10 = pilot; None = all 57 tasks
RUNS    = 3                       # execution runs per task and condition
BELIEF_RUNS = 3                   # belief runs per task and condition
LENGTHS = [2, 5]                  # history lengths -> H2 and H5
# ----------------------------------------------------------------

NAME  = MODEL.split("/")[-1].lower()
TAG   = f"{NAME}_{'pilot' if LIMIT else 'full'}"
LOCAL = f"/content/results/{TAG}"                                   # everything is written here first
DRIVE = f"/content/drive/MyDrive/risk_discounting/sprint/{TAG}"     # copied here every 15 min and after each step
os.makedirs(LOCAL, exist_ok=True); os.makedirs(DRIVE, exist_ok=True)
if any(os.scandir(DRIVE)):
    shutil.copytree(DRIVE, LOCAL, dirs_exist_ok=True)               # carry on from an earlier session
print(MODEL, "| limit:", LIMIT, "| runs:", RUNS, "| GPU:", os.popen("nvidia-smi -L").read().strip() or "none")
print("local:", LOCAL, "\ndrive:", DRIVE)

## Step 0b. Keep Drive in step with the local folder
Starts a background copy every 15 minutes and gives you `sync()`, which every step below calls when it finishes. Nothing is ever deleted on Drive.

**Check:** it prints the first copy's file count. Call `sync()` by hand any time.

In [ ]:
def sync(quiet=True):
    """Copy new or changed files from the local results folder to Drive. Nothing on Drive is deleted."""
    copied = 0
    for src in Path(LOCAL).rglob("*"):
        if src.is_dir():
            continue
        dst = Path(DRIVE) / src.relative_to(LOCAL)
        if dst.exists() and dst.stat().st_size == src.stat().st_size and dst.stat().st_mtime >= src.stat().st_mtime:
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied += 1
    if not quiet:
        print(time.strftime("%H:%M:%S"), f"synced to Drive: {copied} file(s) copied")
    return copied

def _loop():
    while True:
        time.sleep(900)          # 15 minutes
        try:
            sync()
        except Exception as e:
            print("sync failed:", e)

if not globals().get("_SYNCING"):
    threading.Thread(target=_loop, daemon=True).start()
    _SYNCING = True
sync(quiet=False)

## Step 1. Choose the tasks
Keeps the tasks that have at least 5 usable histories, so the same tasks can be compared in all three conditions. No task is dropped for being unsafe in BASE.

**Check:** the table of how many tasks each history length would give, and the last line (`kept N tasks`). The pilot keeps the first `LIMIT` of them.

In [ ]:
limit = f"--limit {LIMIT}" if LIMIT else ""
!python scripts/1_select_tasks.py --long {max(LENGTHS)} {limit} 2>&1 | tee {LOCAL}/step1_select.log
sync()

## Step 2. Build BASE, H2 and H5
Adds the risk line to every condition and the history block to H2 and H5. The short history is the first part of the long one.

**Check:** each condition reports the same number of tasks, and the printed sample prompts read correctly: BASE has the risk and no history, H2 has two experiences, H5 has five.

In [ ]:
!python scripts/2_build_conditions.py --lengths {" ".join(map(str, LENGTHS))} 2>&1 | tee {LOCAL}/step2_conditions.log | head -60
sync()

## Step 3. Belief run: what it says it would do
No tools. It answers Risk / Reasoning / Plan, and its `<think>` block is saved with the answer.

**Writes** `belief/<condition>.jsonl`, a row per task and run as each comes back.
**Check:** `read N of M answers` should be at least 95%, and `thinking saved` should be close to 1.00 — if it is 0.00 the model is not returning `<think>` and step 6 will have less to work with. Run the cell again to retry only the rows that failed.

In [ ]:
!python scripts/3_run_belief.py --model {MODEL} --out {LOCAL} --runs {BELIEF_RUNS} 2>&1 | tee {LOCAL}/step3_belief.log | tail -20
sync(quiet=False)

## Step 4. Execution run: what it does
The same prompts, with tools. One model server, loaded once; every (condition, task, run) is a job. Nothing is added to the prompt, so the thinking is what the agent produced while working.

**Writes** `agent/<condition>/run_<k>/generated_trajectories.json`.
**Check:** the job count is 3 conditions x tasks x `RUNS`, the model loads once at the start, and the end says how many failed (0 is expected).

In [ ]:
!python scripts/4_run_agent.py --model {MODEL} --out {LOCAL} --runs {RUNS} 2>&1 | tee {LOCAL}/step4_agent.log | tail -20
sync(quiet=False)

## Step 5. Judge with ShieldAgent
The benchmark's own judge marks every run safe or unsafe. It loads after the agent's servers have stopped, so run this only once step 4 has finished.

**Writes** `judged/<condition>/run_<k>/`.
**Check:** it prints `judging N: BASE run_1, H2 run_1, ...` and ends with `done`. A folder that is already judged is skipped.

In [ ]:
!python scripts/5_judge.py --out {LOCAL} 2>&1 | tee {LOCAL}/step5_judge.log | tail -15
sync(quiet=False)

## Step 6. Compare what it said with what it did
Claude Haiku reads the belief answer and the execution trajectory for the same task and condition, and returns three yes/no fields plus a one-line note.

**Writes** `compare/<condition>.jsonl`.
**Check:** `answered N of M`, then the per-condition line. `followed its plan` should fall and `ignored a precaution` should rise from BASE to H2 to H5 if the hypothesis holds.

In [ ]:
!python scripts/6_compare.py --out {LOCAL} 2>&1 | tee {LOCAL}/step6_compare.log | tail -15
sync(quiet=False)

## Step 7. The result
Shares per condition and the change against BASE, paired by task, with 95% bootstrap intervals over tasks.

**Writes** `analysis/result.txt`, `result.json`, `per_task.csv`.
**Check:** `knew the risk` should stay flat while `knew and unsafe` and `ignored a precaution` rise. An interval that includes 0 means the sample is too small to call it.

In [ ]:
!python scripts/7_analyze.py --out {LOCAL} 2>&1 | tee {LOCAL}/step7_result.log
sync(quiet=False)

## Step 8. Read some examples by hand
Prints the cases where it said the task was risky, named a precaution, and then skipped it. Read a few to check that Haiku's labels are right, and quote one or two in the report.

In [ ]:
import glob
rows = [json.loads(l) for f in sorted(glob.glob(f"{LOCAL}/compare/H*.jsonl")) for l in open(f) if l.strip()]
belief = {(r["condition"], r["task"]): r for f in sorted(glob.glob(f"{LOCAL}/belief/*.jsonl")) for l in open(f) if l.strip() for r in [json.loads(l)] if r.get("parsed")}
bad = [r for r in rows if r.get("parsed") and r.get("ignored_precaution")][:5]
for r in bad:
    b = belief.get((r["condition"], r["task"]), {})
    print(f"=== {r['condition']} task {r['task']} run {r['run']}")
    print("SAID  risk:", b.get("risk"), "| plan:", (b.get("plan") or "")[:200])
    print("DID   note:", r.get("note"))
    print("THINKING:", (r.get("agent_think") or "")[:400], "\n" + "-" * 70)
print(len(bad), "examples shown;", len([r for r in rows if r.get("ignored_precaution")]), "in total")

## Step 9. Final save
Copies everything to Drive and lists what is there.

In [ ]:
sync(quiet=False)
for p in sorted(Path(DRIVE).rglob("*")):
    if p.is_file() and p.suffix in (".txt", ".json", ".csv", ".log"):
        print(p.relative_to(DRIVE), f"{p.stat().st_size / 1024:.0f} KB")